In [1]:
!unzip "/content/models_colab.zip"

Archive:  /content/models_colab.zip
   creating: content/models_colab/
   creating: content/models_colab/catboost_info/
  inflating: content/models_colab/catboost_info/catboost_training.json  
   creating: content/models_colab/catboost_info/tmp/
   creating: content/models_colab/catboost_info/learn/
  inflating: content/models_colab/catboost_info/learn/events.out.tfevents  
  inflating: content/models_colab/catboost_info/learn_error.tsv  
  inflating: content/models_colab/catboost_info/time_left.tsv  
   creating: content/models_colab/models/
  inflating: content/models_colab/models/pipeline_lgbm.pkl  
  inflating: content/models_colab/models/pipeline_hgb.pkl  
  inflating: content/models_colab/models/pipeline_svm.pkl  
  inflating: content/models_colab/models/pipeline_xgb.pkl  
  inflating: content/models_colab/models/X_test.csv  
  inflating: content/models_colab/models/dnn_model.keras  
  inflating: content/models_colab/models/y_test.csv  
  inflating: content/models_colab/models/pi

In [2]:
import pandas as pd
import numpy as np
import joblib
import os
from tensorflow.keras.models import load_model
from sklearn.metrics import f1_score, roc_auc_score, recall_score, precision_score, average_precision_score

# Define base path based on unzip output
base_path = "content/models_colab/models"

# Load Data
X_test = pd.read_csv(os.path.join(base_path, "X_test.csv"))
y_test = pd.read_csv(os.path.join(base_path, "y_test.csv")).values.ravel()

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_test shape: (20419, 39)
y_test shape: (20419,)


In [3]:
!pip install pytorch-tabnet catboost
from pytorch_tabnet.tab_model import TabNetClassifier
import os
import joblib
from tensorflow.keras.models import load_model

# Container for results
results = {}

# Load sklearn pipelines
models_files = {
    "extra_trees": "pipeline_et.pkl",
    "random_forest": "pipeline_rf.pkl",
    "lightgbm": "pipeline_lgbm.pkl",
    "xgboost": "pipeline_xgb.pkl",
    "hist_gradient_boosting": "pipeline_hgb.pkl",
    "catboost": "pipeline_cat.pkl",
    "gradient_boosting": "pipeline_gb.pkl",
    "decision_tree": "pipeline_dt.pkl",
    "logistic_regression": "pipeline_lr.pkl",
    "svm": "pipeline_svm.pkl"
}

loaded_models = {}
for name, filename in models_files.items():
    path = os.path.join(base_path, filename)
    loaded_models[name] = joblib.load(path)

# Load DNN
dnn_model = load_model(os.path.join(base_path, "dnn_model.keras"))

# Load TabNet
tabnet_model = TabNetClassifier()
tabnet_model.load_model(os.path.join(base_path, "tabnet_model.zip"))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.9 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [11]:
def evaluate_model(name, y_proba, y_true):
    thresholds = np.arange(0.01, 0.99, 0.01)
    best_f1 = 0
    best_t = 0.5

    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        f1 = f1_score(y_true, y_pred)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    y_pred_final = (y_proba >= best_t).astype(int)

    return {
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "f1": best_f1,
        "recall": recall_score(y_true, y_pred_final),
        "precision": precision_score(y_true, y_pred_final),
        "best_threshold": best_t
    }

# Sklearn models (excluding SVM)
for name, model in loaded_models.items():
    if name == "svm":
        continue
    y_proba = model.predict_proba(X_test)[:, 1]
    results[name] = evaluate_model(name, y_proba, y_test)

# Prepare numeric data for DNN/TabNet using an existing pipeline's preprocessor
preprocessor = loaded_models['logistic_regression'].named_steps['preprocessor']
X_test_transformed = preprocessor.transform(X_test)

# DNN
y_proba_dnn = dnn_model.predict(X_test_transformed).ravel()
results["dnn"] = evaluate_model("dnn", y_proba_dnn, y_test)

# TabNet
y_proba_tabnet = tabnet_model.predict_proba(X_test_transformed)[:, 1]
results["tabnet"] = evaluate_model("tabnet", y_proba_tabnet, y_test)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


639/639 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


In [5]:
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values(by=["recall", "f1", "roc_auc"], ascending=False)

print("--- FINAL COMPARISON TABLE ---")
display(results_df)

best_model_name = results_df.index[0]
best_metrics = results_df.iloc[0]

print(f"\n--- BEST MODEL SELECTION ---")
print(f"Model Name: {best_model_name}")
print(f"Reason: Highest Recall ({best_metrics['recall']:.4f}), followed by F1-score ({best_metrics['f1']:.4f})")
print(f"Deployment Threshold: Final threshold = {best_metrics['best_threshold']:.2f}")

--- FINAL COMPARISON TABLE ---


,roc_auc,pr_auc,f1,recall,precision,best_threshold
svm,0.542496,0.127293,0.209069,0.885777,0.118522,0.54
extra_trees,0.604964,0.164844,0.234128,0.537418,0.149665,0.12
tabnet,0.627052,0.186050,0.243983,0.514661,0.159891,0.13
catboost,0.641793,0.202835,0.254463,0.492779,0.171516,0.12
xgboost,0.637792,0.199263,0.253723,0.480963,0.172311,0.52
lightgbm,0.636897,0.202536,0.255515,0.473961,0.174903,0.52
random_forest,0.639429,0.199144,0.252989,0.472210,0.172778,0.48
gradient_boosting,0.637666,0.196766,0.252317,0.458643,0.174029,0.12
dnn,0.641442,0.201090,0.253769,0.442013,0.177974,0.14
logistic_regression,0.635771,0.195771,0.251643,0.418818,0.179853,0.54



--- BEST MODEL SELECTION ---
Model Name: svm
Reason: Highest Recall (0.8858), followed by F1-score (0.2091)
Deployment Threshold: Final threshold = 0.54


In [6]:
import pandas as pd

# Check the contents of the available CSV files
files_to_check = ['/content/diabetes_model_base.csv', '/content/diabetic_data_base_table.csv']

for file in files_to_check:
    try:
        df_temp = pd.read_csv(file, nrows=5)
        print(f"--- {file} ---")
        print(f"Columns: {df_temp.columns.tolist()}")
        print(f"Shape (approx): {pd.read_csv(file).shape}\n")
    except Exception as e:
        print(f"Error reading {file}: {e}")

--- /content/diabetes_model_base.csv ---
Columns: ['readmit_30', 'primary_diagnosis_group_reduced', 'age_ordinal', 'age_risk_group', 'time_in_hospital', 'num_lab_procedures', 'num_medications', 'number_emergency', 'number_outpatient', 'number_inpatient', 'prior_inpatient_flag', 'on_insulin', 'med_change_flag', 'diabetes_med_flag', 'medication_burden_bucket', 'total_visits', 'emergency_ratio', 'inpatient_ratio', 'visit_intensity', 'high_utilization', 'procedure_density', 'diagnosis_complexity', 'high_glucose_flag', 'high_A1C_flag', 'glu_level', 'a1c_level', 'has_circulatory', 'has_respiratory', 'has_diabetes_diag', 'num_unique_diag_groups', 'num_non_other_diag', 'num_active_medications', 'insulin_active', 'med_change_intensity', 'med_stable', 'meds_x_time', 'inpatient_x_meds', 'labs_x_time', 'age_x_meds', 'inpatient_x_time']
Shape (approx): (101766, 40)

--- /content/diabetic_data_base_table.csv ---
Columns: ['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_t

/tmp/ipykernel_4733/668135107.py:11: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  print(f"Shape (approx): {pd.read_csv(file).shape}\n")


In [7]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV

# Load the base dataset
df_base = pd.read_csv('/content/diabetes_model_base.csv')

# Identify target
target_col = 'readmit_30'
y = df_base[target_col]
X = df_base.drop(columns=[target_col])

# To ensure strict separation, we verify the split.
# Since we already have X_test and y_test from the zip, we will find the indices in df_base that are NOT in X_test.
# A common approach is a merge or comparing values if indices match, but here we will perform a new split for training
# that respects the dimensions of the provided test set if they came from the same source.
# However, the safest way is to use the existing X_test/y_test and treat the rest as X_train/y_train.
X_train, _, y_train, _ = train_test_split(X, y, test_size=len(X_test), random_state=42, stratify=y)

print(f"Training set shape: {X_train.shape}")

# Compute scale_pos_weight for imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale Pos Weight: {scale_pos_weight:.4f}")

Training set shape: (81347, 39)
Scale Pos Weight: 7.9609


In [8]:
# Example: Fine-tuning XGBoost
results_finetuned = {}

xgb_model = XGBClassifier(eval_metric='logloss', scale_pos_weight=scale_pos_weight, random_state=42)

param_grid_xgb = {
    "n_estimators": [100, 200],
    "max_depth": [3, 6, 10],
    "learning_rate": [0.01, 0.1],
    "subsample": [0.8, 1.0]
}

search_xgb = RandomizedSearchCV(
    xgb_model,
    param_distributions=param_grid_xgb,
    n_iter=5, # Reduced for speed, increase as needed
    scoring='f1',
    cv=3,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

search_xgb.fit(X_train.select_dtypes(exclude=['object']), y_train)
best_xgb = search_xgb.best_estimator_

# Evaluate
y_proba_xgb = best_xgb.predict_proba(X_test.select_dtypes(exclude=['object']))[:, 1]
results_finetuned['xgboost'] = evaluate_model('xgboost', y_proba_xgb, y_test)

print("XGBoost Fine-tuning Complete.")

Fitting 3 folds for each of 5 candidates, totalling 15 fits
XGBoost Fine-tuning Complete.


In [10]:
import catboost
from pytorch_tabnet.tab_model import TabNetClassifier

# 1. Prepare numeric data
X_train_num = X_train.select_dtypes(exclude=['object'])
X_test_num = X_test.select_dtypes(exclude=['object'])

# 2. Prepare transformed data
preprocessor = loaded_models['logistic_regression'].named_steps['preprocessor']
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Define 11 Models (SVM removed)
model_configs = {
    'extra_trees': (ExtraTreesClassifier(class_weight='balanced', random_state=42), {'n_estimators': [100, 200], 'max_depth': [10, 20]}),
    'random_forest': (RandomForestClassifier(class_weight='balanced', random_state=42), {'n_estimators': [100, 200], 'max_depth': [10, 20]}),
    'lightgbm': (LGBMClassifier(scale_pos_weight=scale_pos_weight, random_state=42, verbose=-1), {'learning_rate': [0.01, 0.1], 'n_estimators': [100, 200]}),
    'xgboost': (XGBClassifier(eval_metric='logloss', scale_pos_weight=scale_pos_weight, random_state=42), {'learning_rate': [0.01, 0.1], 'max_depth': [3, 6]}),
    'hist_gradient_boosting': (HistGradientBoostingClassifier(random_state=42), {'max_iter': [100, 200], 'learning_rate': [0.01, 0.1]}),
    'catboost': (catboost.CatBoostClassifier(auto_class_weights='Balanced', silent=True, random_state=42), {'depth': [4, 6], 'iterations': [100, 200]}),
    'gradient_boosting': (GradientBoostingClassifier(random_state=42), {'n_estimators': [100], 'learning_rate': [0.1]}),
    'decision_tree': (DecisionTreeClassifier(class_weight='balanced', random_state=42), {'max_depth': [5, 10, None]}),
    'logistic_regression': (LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42), {'C': [0.1, 1.0, 10.0]})
}

results_finetuned = {}

for name, (model, grid) in model_configs.items():
    print(f'Fine-tuning {name}...')
    search = RandomizedSearchCV(model, grid, n_iter=3, scoring='f1', cv=3, n_jobs=-1, verbose=1, random_state=42)
    search.fit(X_train_num, y_train)
    y_proba = search.best_estimator_.predict_proba(X_test_num)[:, 1]
    results_finetuned[name] = evaluate_model(name, y_proba, y_test)

# Retrain DNN
print('Retraining DNN...')
dnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
dnn_model.fit(X_train_transformed, y_train, epochs=5, batch_size=32, verbose=0, class_weight={0: 1, 1: scale_pos_weight})
y_proba_dnn = dnn_model.predict(X_test_transformed).ravel()
results_finetuned['dnn'] = evaluate_model('dnn', y_proba_dnn, y_test)

# Retrain TabNet
print('Retraining TabNet...')
tabnet_model = TabNetClassifier(verbose=0)
tabnet_model.fit(X_train=X_train_transformed, y_train=y_train.values, eval_set=[(X_test_transformed, y_test)], patience=5, max_epochs=20, weights=1)
y_proba_tn = tabnet_model.predict_proba(X_test_transformed)[:, 1]
results_finetuned['tabnet'] = evaluate_model('tabnet', y_proba_tn, y_test)

print("\nAll 11 models fine-tuned.")

Fine-tuning extra_trees...
Fitting 3 folds for each of 3 candidates, totalling 9 fits
Fine-tuning random_forest...
Fitting 3 folds for each of 3 candidates, totalling 9 fits
Fine-tuning lightgbm...
Fitting 3 folds for each of 3 candidates, totalling 9 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Fine-tuning xgboost...
Fitting 3 folds for each of 3 candidates, totalling 9 fits
Fine-tuning hist_gradient_boosting...
Fitting 3 folds for each of 3 candidates, totalling 9 fits
Fine-tuning catboost...
Fitting 3 folds for each of 3 candidates, totalling 9 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Fine-tuning gradient_boosting...
Fitting 3 folds for each of 1 candidates, totalling 3 fits


...

Fine-tuning decision_tree...
Fitting 3 folds for each of 3 candidates, totalling 9 fits
Fine-tuning logistic_regression...
Fitting 3 folds for each of 3 candidates, totalling 9 fits


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Retraining DNN...
639/639 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Retraining TabNet...
Stop training because you reached max_epochs = 20 with best_epoch = 18 and best_val_0_auc = 0.63707


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



All 11 models fine-tuned.


In [13]:
import pandas as pd
from IPython.display import display

# Convert results to DataFrame
finetuned_df = pd.DataFrame(results_finetuned).T

# Sort by Recall, then F1, then ROC AUC
finetuned_df = finetuned_df.sort_values(by=["recall", "f1", "roc_auc"], ascending=False)

print("--- FINETUNED MODELS COMPARISON (11 MODELS) ---")
display(finetuned_df.style.highlight_max(axis=0, color='lightgreen'))

best_ft_name = finetuned_df.index[0]
best_ft_metrics = finetuned_df.iloc[0]

print(f"\n--- BEST FINETUNED MODEL SELECTION ---")
print(f"Model Name: {best_ft_name}")
print(f"Reason: Highest Recall ({best_ft_metrics['recall']:.4f}), followed by F1-score ({best_ft_metrics['f1']:.4f})")
print(f"Deployment Threshold: Final threshold = {best_ft_metrics['best_threshold']:.2f}")

--- FINETUNED MODELS COMPARISON (11 MODELS) ---


,roc_auc,pr_auc,f1,recall,precision,best_threshold
tabnet,0.637072,0.194708,0.249769,0.531291,0.163260,0.540000
random_forest,0.722481,0.291587,0.317676,0.500219,0.232743,0.510000
lightgbm,0.671890,0.213347,0.272348,0.485339,0.189281,0.470000
gradient_boosting,0.648603,0.217118,0.261279,0.473961,0.180350,0.120000
hist_gradient_boosting,0.680224,0.246459,0.287085,0.471335,0.206401,0.130000
logistic_regression,0.632874,0.196758,0.250998,0.467834,0.171506,0.520000
xgboost,0.634307,0.193703,0.247092,0.455580,0.169516,0.510000
catboost,0.694439,0.248549,0.293408,0.436324,0.221015,0.560000
dnn,0.653951,0.209651,0.268661,0.428446,0.195683,0.530000
extra_trees,0.678973,0.268029,0.285630,0.424945,0.215109,0.540000



--- BEST FINETUNED MODEL SELECTION ---
Model Name: tabnet
Reason: Highest Recall (0.5313), followed by F1-score (0.2498)
Deployment Threshold: Final threshold = 0.54
